# One Ride Through GoRide

### Cloud Pub/Sub, told as a story - one cell at a time

We follow **one ride** through a ride company called **GoRide**: Maya finishes a $24.50 trip
across San Francisco, the app publishes **one event**, and we watch the whole company react -
dispatch, finance, and the data team's BigQuery dashboard - each independently.

```
GoRide app ──>  rides topic  ──┬─> match-sub     > Dispatch
   (Maya)      (Avro schema)   ├─> billing-sub   > Finance
                               └─> analytics-sub > BigQuery (data team)
```

**Prerequisite:** the infrastructure exists - run `terraform apply` first (see `SETUP.md`).
This notebook is the *live demo surface*; Terraform is the stagehand.

**Where to run this:** Cloud Shell Editor, Vertex AI Workbench, or BigQuery Studio (auth is
automatic in all three). In Colab, uncomment the `auth.authenticate_user()` line below.

In [ ]:
# Setup - the only install this demo needs.
%pip install --quiet google-cloud-pubsub google-cloud-bigquery pandas db-dtypes

In [ ]:
import json
import time

import google.auth
from google.cloud import pubsub_v1

# In Colab only, uncomment these two lines first:
# from google.colab import auth
# auth.authenticate_user()

# google.auth.default() returns a (credentials, project_id) pair.
# We only need the project ID here; auth itself is automatic in
# Cloud Shell / Vertex / BigQuery Studio.
credentials, PROJECT_ID = google.auth.default()
# PROJECT_ID = "your-project-id"  # <- uncomment to override

TOPIC_ID = "rides"

print(f"Project: {PROJECT_ID}")
print(f"Topic:   {TOPIC_ID}")

---
## Scene 1 - Friday night at GoRide

It's Friday night in San Francisco. **Maya** opens the GoRide app and requests a ride. That one
tap has to reach **dispatch** (find her a driver), **finance** (get ready to charge her card),
and the **data team's** live dashboards.

The old GoRide architecture did this with **direct calls** - the trip service called dispatch,
then billing, then analytics, one by one:

- Maya stared at a spinner while every call finished
- the night payments went down, *ride requests* started failing - one failure cascaded
- every new consumer (fraud? notifications?) meant editing the trip service's core code

GoRide fixed it with one architectural move: the app now **publishes each trip event once** to
Pub/Sub, and everyone who cares **subscribes**. Let's tour tonight's setup - then Maya takes
her ride.

---
## Scene 2 - Tour of the machinery

One topic, three subscriptions. Note what's **absent**: the publisher has no list of consumers
anywhere. Dispatch, finance, and the data team each opened their own subscription - the app has
no idea they exist.

In [ ]:
publisher = pubsub_v1.PublisherClient()

# Resources have full names like projects/<project>/topics/<topic>.
# topic_path() just builds that string for us.
topic_path = publisher.topic_path(PROJECT_ID, TOPIC_ID)

print(f"Topic: {topic_path}\n")
print("Subscriptions fanning out from it:")
for sub in publisher.list_topic_subscriptions(request={"topic": topic_path}):
    # sub is the full name (projects/.../subscriptions/match-sub);
    # keep just the short name after the last "/".
    print("  •", sub.split("/")[-1])

The topic also carries an **Avro schema** - GoRide's company-wide data contract. Every message
must have exactly these seven fields, with `fare` as a number. We'll watch it earn its keep in
Scene 6.

---
## Scene 3 - Maya's trip completes *(the live moment)*

Maya's ride just ended - **$24.50 across town**. Right now, *this notebook is the GoRide app*,
and it does the one thing the app does: **publish the event.**

Notice how fast the cell finishes - the publish returns the moment Pub/Sub has stored the
message. The app doesn't wait for dispatch, finance, or analytics.
**The publisher never blocks on the rest of the company.**

In [ ]:
maya_event = {
    "event_id": "evt_maya",
    "user_id": "u_456",
    "driver_id": "d_789",
    "event_type": "trip_completed",
    "fare": 24.50,
    "city": "San Francisco",
    "timestamp": "2026-07-22T22:15:00Z",
}

# The topic's schema uses JSON encoding, so the message body is JSON bytes.
data = json.dumps(maya_event).encode("utf-8")

future = publisher.publish(topic_path, data)   # <- the app's entire job
message_id = future.result()                   # done as soon as Pub/Sub has stored it

print(f"Published Maya's trip - message ID {message_id}")
print("Maya's phone shows 'trip complete'. She's already walking away.")
print("But three departments just got word. Let's go visit them...")

---
## Scene 4 - Dispatch and Finance both react

First stop: **dispatch's inbox** (`match-sub`). Note the order in the code - **process FIRST,
ack AFTER**. If dispatch crashed mid-processing, the un-acked message would be redelivered and
no trip would ever be lost. Ack means exactly one thing: *"I processed this successfully."*

In [ ]:
subscriber = pubsub_v1.SubscriberClient()
match_path = subscriber.subscription_path(PROJECT_ID, "match-sub")

time.sleep(3)  # give delivery a moment to catch up with the publish

# Ask Pub/Sub for up to 5 waiting messages.
# (The client takes its arguments as one `request` dictionary -
#  it reads oddly at first, but it is exactly how Google's docs write it.)
response = subscriber.pull(request={"subscription": match_path, "max_messages": 5})

for received in response.received_messages:
    event = json.loads(received.message.data)

    # 1. PROCESS first...
    print(f"[dispatch] {event['event_type']}: rider {event['user_id']} -> driver {event['driver_id']} freed")

    # 2. ...then ACK: 'processed successfully - stop redelivering this.'
    subscriber.acknowledge(request={"subscription": match_path, "ack_ids": [received.ack_id]})
    print(f"   ack ({event['event_id']})")

Now **finance** (`billing-sub`). Watch what comes back: **the same event, again** - finance's
*own copy*. That's **fan-out**: every subscription gets its own copy of every message.

Finance never talked to dispatch. If finance's systems were down for an hour, their copy would
wait here - and dispatch wouldn't notice a thing. And when the fraud team finally wants in?
**One new subscription. Zero changes** to the app, dispatch, or finance.

*(Contrast: two workers on ONE subscription would **split** the messages - that's
load-balancing, for scaling a single job. Separate subscriptions = separate jobs.)*

In [ ]:
billing_path = subscriber.subscription_path(PROJECT_ID, "billing-sub")

response = subscriber.pull(request={"subscription": billing_path, "max_messages": 5})

for received in response.received_messages:
    event = json.loads(received.message.data)
    print(f"[finance] charging {event['user_id']} ${event['fare']:.2f} for {event['event_id']} - receipt sent")
    subscriber.acknowledge(request={"subscription": billing_path, "ack_ids": [received.ack_id]})
    print("   ack - the SAME event dispatch already handled. Their own copy.")

### How dispatch runs this in production *(optional beat)*

Pulling in a loop is fine for a notebook, but a real dispatch service uses **streaming pull**:
register a **callback function**, and Pub/Sub calls it the instant a message arrives.

Run the next cell - it listens for 15 seconds, then stops itself. While it's listening,
**re-run the Scene 3 publish cell** (give it a new `event_id`) and watch dispatch react in
real time.

In [ ]:
def callback(message):
    event = json.loads(message.data)
    print(f"[dispatch-service] {event['event_id']} arrived -> processing... ack")
    message.ack()  # process first in real code - then ack

# subscribe() returns immediately; Pub/Sub runs the callback in the background
# every time a message arrives.
listener = subscriber.subscribe(match_path, callback=callback)
print("dispatch-service listening on match-sub - publish another event now!\n")

time.sleep(15)      # stay alive and let the callback do its work
listener.cancel()   # stop listening
print("\ndispatch-service stopped listening.")

---
## Scene 5 - The data team already sees it

While we were visiting dispatch and finance, the third subscriber never needed a person - or
any code. `analytics-sub` is a **BigQuery subscription**: Pub/Sub streams every event straight
into the `rides_analytics.trip_events` table. No ETL job, no pipeline to babysit.

Let's check the data team's warehouse. **Maya's ride should already be a row.**

In [ ]:
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID)

sql = f"""
    SELECT event_id, event_type, city, fare, timestamp
    FROM `{PROJECT_ID}.rides_analytics.trip_events`
    ORDER BY timestamp DESC
    LIMIT 20
"""
bq.query(sql).to_dataframe()
# If evt_maya isn't in the result yet, re-run this cell - BigQuery subscriptions
# stream continuously and land within seconds.

This is the deck's *"one trip → one smart notification"* hop: the same event that drove
operations also feeds analytics, in real time. GoRide's surge dashboard - and that
*"10th trip this month → loyalty push"* flow - starts exactly here.

---
## Scene 6 - Something goes wrong. Nothing breaks.

It's still Friday night, so of course something goes wrong: a **partner integration ships a
bug** and sends `fare` as *text*.

The topic's Avro schema is about to bounce it at the front door - dispatch, finance, and the
dashboard never see garbage. **This cell is supposed to fail.**

In [ ]:
bad_event = {
    "event_id": "evt_bad",
    "user_id": "u_999",
    "driver_id": "d_000",
    "event_type": "trip_completed",
    "fare": "not-a-number",          # ← the bug: fare must be a double
    "city": "Nowhere",
    "timestamp": "2026-07-22T22:20:00Z",
}

try:
    publisher.publish(topic_path, json.dumps(bad_event).encode("utf-8")).result()
    print("This should not have been accepted!")
except Exception as err:
    print("REJECTED at publish time - the schema held the line:\n")
    print(f"   {err}")

And the second failure mode: if a message gets *through* but dispatch keeps crashing on it,
`match-sub`'s **dead-letter policy** routes it to the `rides-dead-letter` topic after 5 failed
deliveries - quarantined for an engineer to inspect Monday morning, while the Friday-night rush
keeps flowing behind it. *(It's configuration, not code - see `terraform/main.tf`.)*

---
## The closer

> One tap. **One publish.** Dispatch reacted, finance reacted, the dashboard updated - three
> departments, each at their own pace, each with their own copy. A corrupted event got bounced,
> and a poison message would be quarantined - none of it touching Maya's experience.
>
> The GoRide app knows about **none** of this. It published once, and everything reacted.
>
> **That's Cloud Pub/Sub: publish once. Let everything react.**

---

**Cleanup** (after the talk, in a terminal - not here):

```bash
cd terraform && terraform destroy
```